# GDN Benchmark on Google Colab

This notebook runs the repository's clean-room GDN adapter with the common validation-only thresholding and point-wise evaluation utilities.

## Data and outputs

- Repository cloned or updated from: `https://github.com/Yan9564/INFORMS_FORD.git`
- Dataset root: `/content/drive/MyDrive/anomaly_benchmark_data`
- Result root: `/content/drive/MyDrive/anomaly_benchmark_results/gdn/`

## Execution modes

- `smoke`: small subset, one seed, few epochs, CPU/GPU auto-detect.
- `full`: complete configured dataset, multiple seeds, more epochs.

## Notes

SWaT is the primary temporal benchmark for this notebook. Ford is supported only when the researcher confirms the CSV rows are chronologically meaningful and should be evaluated as temporal windows. Test labels are never used for training, hyperparameter selection, or threshold selection.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, platform, time, traceback, random, shutil
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
REPO_URL = 'https://github.com/Yan9564/INFORMS_FORD.git'
BRANCH = 'main'
DATASET_ROOT = Path('/content/drive/MyDrive/anomaly_benchmark_data')
RESULT_ROOT = Path('/content/drive/MyDrive/anomaly_benchmark_results/gdn')
MODE = 'smoke'  # change to 'full' for archival runs
DATASET = 'swat'  # 'swat' primary; 'ford' only after chronological-order confirmation
print('In Colab:', IN_COLAB)
print(sys.version)
print(platform.platform())


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Local runtime detected; Google Drive mount skipped.')


In [ ]:
repo_dir = Path('/content/INFORMS_FORD') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if repo_dir.exists():
        subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
print('Repository:', Path.cwd())


In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .[gdn]


In [ ]:
import numpy as np
import pandas as pd
import yaml

from benchmark.metrics.classification import evaluate_binary
from benchmark.metrics.thresholding import select_threshold
from benchmark.models.gdn import GDNDetector

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    DEVICE = 'cpu'
    print('Torch import failed:', exc)


In [ ]:
MODE_CONFIG = {
    'smoke': {'seeds': [42], 'epochs': 2, 'max_train_rows': 5000, 'max_test_rows': 5000, 'hidden_dim': 32, 'batch_size': 64},
    'full': {'seeds': [42, 43, 44], 'epochs': 20, 'max_train_rows': None, 'max_test_rows': None, 'hidden_dim': 64, 'batch_size': 128},
}
RUN_CFG = MODE_CONFIG[MODE]
LOOKBACK = 10
THRESHOLD_PERCENTILE = 95.0
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(RUN_CFG)


## Dataset loading

For SWaT, place normal training and attack/test CSV files under `anomaly_benchmark_data/swat/`. Update file names and label column below if your copy differs. For Ford, the notebook reuses the existing Ford loader and requires `windowing.representation='sequence'`.


In [ ]:
def make_windows(values, labels=None, lookback=10):
    X, y = [], []
    for start in range(0, len(values) - lookback + 1):
        X.append(values[start:start + lookback])
        if labels is not None:
            y.append(int(labels[start + lookback - 1]))
    if not X:
        raise ValueError('No windows created; reduce LOOKBACK or check data length.')
    return np.asarray(X, dtype=np.float32), (None if labels is None else np.asarray(y, dtype=int))

def load_swat():
    swat_root = DATASET_ROOT / 'swat'
    train_csv = swat_root / 'SWaT_Dataset_Normal_v1.csv'
    test_csv = swat_root / 'SWaT_Dataset_Attack_v0.csv'
    label_column = 'Normal/Attack'
    for path in [train_csv, test_csv]:
        if not path.exists():
            raise FileNotFoundError(f'Missing SWaT file: {path}')
    train_df = pd.read_csv(train_csv, nrows=RUN_CFG['max_train_rows'])
    test_df = pd.read_csv(test_csv, nrows=RUN_CFG['max_test_rows'])
    feature_cols = [c for c in train_df.columns if c in test_df.columns and c != label_column and pd.api.types.is_numeric_dtype(train_df[c])]
    if not feature_cols:
        raise ValueError('No numeric SWaT feature columns found; check CSV parsing and column names.')
    train_values = train_df[feature_cols].apply(pd.to_numeric, errors='coerce').ffill().bfill().fillna(0.0).to_numpy(dtype=np.float32)
    test_values = test_df[feature_cols].apply(pd.to_numeric, errors='coerce').ffill().bfill().fillna(0.0).to_numpy(dtype=np.float32)
    raw_labels = test_df[label_column].astype(str).str.lower() if label_column in test_df else pd.Series(np.zeros(len(test_df), dtype=int))
    test_labels = raw_labels.map(lambda v: 0 if 'normal' in v or v == '0' else 1).to_numpy(dtype=int)
    split = max(LOOKBACK + 1, int(0.8 * len(train_values)))
    X_train, _ = make_windows(train_values[:split], None, LOOKBACK)
    X_val, y_val = make_windows(train_values[split - LOOKBACK + 1:], np.zeros(len(train_values[split - LOOKBACK + 1:]), dtype=int), LOOKBACK)
    X_test, y_test = make_windows(test_values, test_labels, LOOKBACK)
    return X_train, X_val, y_val, X_test, y_test, feature_cols, {'train_csv': str(train_csv), 'test_csv': str(test_csv)}

def load_ford():
    from benchmark.datasets.ford import FordDatasetConfig, FordDatasetLoader
    ford_root = DATASET_ROOT / 'ford' / 'Train'
    cfg = {
        'name': 'ford', 'accepted_dir': str(ford_root / 'accept'), 'rejected_dir': str(ford_root / 'reject'),
        'feature_columns': {'prefix': 'feature_', 'start': 0, 'end': 47}, 'label_column': 'feature_55',
        'normal_label': 0, 'anomaly_label': 1, 'validation_fraction': 0.2, 'split_by_file': True,
        'random_seed': 42, 'missing_values': {'strategy': 'median'},
        'clipping': {'enabled': True, 'lower_quantile': 0.001, 'upper_quantile': 0.999},
        'scaling': {'method': 'standard'}, 'windowing': {'lookback': LOOKBACK, 'stride': 1, 'representation': 'sequence'},
        'downsampling': {'every_n_rows': 1},
    }
    bundle = FordDatasetLoader(FordDatasetConfig.from_dict(cfg)).load()
    return bundle.X_train, bundle.X_validation, bundle.y_validation, bundle.X_test, bundle.y_test, bundle.feature_names, {'ford_root': str(ford_root)}

if DATASET == 'swat':
    X_train, X_val, y_val, X_test, y_test, feature_cols, source_info = load_swat()
elif DATASET == 'ford':
    X_train, X_val, y_val, X_test, y_test, feature_cols, source_info = load_ford()
else:
    raise ValueError(f'Unsupported DATASET: {DATASET}')
print('X_train', X_train.shape, 'X_val', X_val.shape, 'X_test', X_test.shape)
print('features', len(feature_cols), feature_cols[:10])
print('validation labels', dict(zip(*np.unique(y_val, return_counts=True))))
print('test labels', dict(zip(*np.unique(y_test, return_counts=True))))


In [ ]:
def run_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    if 'torch' in globals():
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    run_dir = RESULT_ROOT / DATASET / MODE / f'seed_{seed}'
    metrics_path = run_dir / 'metrics.json'
    if metrics_path.exists():
        print(f'Skipping completed seed {seed}: {metrics_path}')
        return json.loads(metrics_path.read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    failure_path = run_dir / 'failure.log'
    t0 = time.perf_counter()
    try:
        model = GDNDetector(epochs=RUN_CFG['epochs'], learning_rate=0.001, hidden_dim=RUN_CFG['hidden_dim'], batch_size=RUN_CFG['batch_size'], top_k=min(5, X_train.shape[-1]-1), device=DEVICE, random_state=seed)
        train_start = time.perf_counter(); model.fit(X_train); train_runtime = time.perf_counter() - train_start
        score_start = time.perf_counter(); val_scores = model.score_samples(X_val); test_scores = model.score_samples(X_test); score_runtime = time.perf_counter() - score_start
        threshold = select_threshold(val_scores, method='percentile', percentile=THRESHOLD_PERCENTILE)
        metrics = evaluate_binary(y_test, test_scores, threshold.threshold)
        predictions = (test_scores >= threshold.threshold).astype(int)
        runtime = {'train_seconds': train_runtime, 'score_seconds': score_runtime, 'total_seconds': time.perf_counter() - t0, 'device': DEVICE}
        resolved = {'dataset': DATASET, 'mode': MODE, 'seed': seed, 'lookback': LOOKBACK, 'threshold_percentile': THRESHOLD_PERCENTILE, 'source_info': source_info, 'model': model.export_configuration()}
        out = {'dataset': DATASET, 'model': 'gdn', 'seed': seed, 'precision': metrics['precision'], 'recall': metrics['recall'], 'f1': metrics['f1'], 'auroc': metrics['auroc'], 'auprc': metrics['auprc'], 'runtime': runtime, 'evaluation_protocol': 'point-wise/window-final-label; validation-only percentile threshold; no point adjustment', 'warnings': metrics.get('warnings', [])}
        model.save(run_dir / 'checkpoint.pt')
        np.savez_compressed(run_dir / 'scores_predictions.npz', test_scores=test_scores, predictions=predictions, y_test=y_test, validation_scores=val_scores)
        (run_dir / 'training_log.json').write_text(json.dumps(model.training_log_, indent=2))
        (run_dir / 'runtime.json').write_text(json.dumps(runtime, indent=2))
        (run_dir / 'resolved_config.yaml').write_text(yaml.safe_dump(resolved, sort_keys=False))
        (run_dir / 'environment.txt').write_text(f'python={sys.version}
platform={platform.platform()}
torch={getattr(torch, "__version__", "not imported")}
device={DEVICE}
')
        metrics_path.write_text(json.dumps(out, indent=2))
        pd.DataFrame([out]).to_csv(run_dir / 'metrics.csv', index=False)
        failure_path.write_text('')
        return out
    except Exception:
        failure_path.write_text(traceback.format_exc())
        raise

results = [run_seed(seed) for seed in RUN_CFG['seeds']]
summary = pd.DataFrame(results)
summary[['dataset', 'model', 'seed', 'precision', 'recall', 'f1', 'auroc', 'auprc', 'evaluation_protocol']]
